# Exercício Prático — O Pipeline de Machine Learning com o dataset Iris

Neste exercício vamos percorrer, na prática, **todas as etapas do pipeline de Machine Learning** usando o dataset **Iris** (medidas de flores de 3 espécies).

**As etapas do pipeline:**

1. **Definição do problema**
2. **Preparação dos dados**
3. **Divisão treino / teste**
4. **Pré-processamento**
5. **Treinamento do modelo**
6. **Predição**
7. **Avaliação**

> 🔎 A **Visualização** aparece aqui na **exploração dos dados (EDA)**. Ela também é muito usada na *avaliação* dos resultados — isso veremos na próxima aula.  
> 🚀 Ao final, um modelo aprovado pode ser **implantado** para uso em dados novos.


## Configuração — Importar as bibliotecas

`pandas`/`numpy` para os dados, `matplotlib`/`seaborn` para os gráficos e `datasets` do scikit-learn para carregar o Iris. As funções específicas de cada etapa serão importadas na própria etapa.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets

## Etapa 1 — Definição do problema

Queremos **prever a espécie de uma flor Iris** (setosa, versicolor ou virginica) a partir de 4 medidas: comprimento e largura da **sépala** e da **pétala**.

- Como a saída é uma **categoria**, esta é uma tarefa de **classificação**.
- Como os dados têm **rótulos** (sabemos a espécie de cada flor), é **aprendizado supervisionado**.

## Etapa 2 — Preparação dos dados

Carregamos o dataset, organizamos em uma tabela (**DataFrame**) e, mais adiante, separamos as **variáveis de entrada (X)** do **alvo (y)**.

In [ ]:
# Carrega o dataset Iris (já vem com o scikit-learn)
data = datasets.load_iris()

# Organiza os dados em uma tabela: uma coluna por medida
df = pd.DataFrame(data.data, columns=data.feature_names)

# Rótulo numérico da espécie (0, 1, 2)
df['target'] = data.target

# Nome da espécie (facilita a leitura e os gráficos)
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

Uma olhada nas primeiras linhas e nas estatísticas descritivas:

In [ ]:
# Primeiras linhas da tabela
df.head()
# Sepal length = Comprimento da sépala
# Sepal width = Largura da sépala
# Petal length = Comprimento da pétala
# Petal width = Largura da pétala

In [ ]:
# Estatísticas descritivas de cada variável
df.describe()

### 🔎 Visualização (etapa transversal) — Exploração dos dados (EDA)

Antes de modelar, é essencial **olhar os dados**. Os **histogramas** mostram como cada variável se distribui; os **gráficos de dispersão** mostram como as espécies se separam no espaço das medidas — o que ajuda a escolher e a confiar no modelo.

In [ ]:
# Distribuição de cada medida
df[data.feature_names].hist(figsize=(10, 7), bins=15, edgecolor='black')
plt.suptitle('Distribuição das variáveis', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Relação entre as variáveis, colorindo por espécie
sns.pairplot(df, vars=data.feature_names, hue='species', markers=['o', 's', 'D'])
plt.suptitle('Dispersão por espécie', y=1.02)
plt.show()

### Separar as variáveis: entradas (X) e alvo (y)

`X` contém as 4 medidas (features) e `y` contém a espécie a prever (0, 1, 2).

In [ ]:
X = df[data.feature_names]   # variáveis de entrada (features)
y = df['target']             # alvo a prever (rótulo)

print('X:', X.shape, '| y:', y.shape)

## Etapa 3 — Divisão em treino e teste

Separamos parte dos dados para **avaliar o modelo em exemplos que ele não viu**.

- `test_size=0.2` → 20% para teste, 80% para treino.
- `random_state=42` → fixa a aleatoriedade (qualquer número serve); garante a **mesma divisão** toda vez.
- `stratify=y` → mantém a **proporção das 3 espécies** no treino e no teste.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Treino:', X_train.shape[0], 'amostras | Teste:', X_test.shape[0], 'amostras')

## Etapa 4 — Pré-processamento (padronização)

Colocamos as variáveis na **mesma escala** (média 0, desvio 1). Ajustamos o `StandardScaler` **só no treino** (`fit_transform`) e aplicamos a mesma escala no teste (`transform`), para não "vazar" informação do teste. Isso é importante para o KNN, que usa **distâncias**.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # aprende a escala no treino e aplica
X_test = scaler.transform(X_test)         # aplica a MESMA escala no teste

## Etapa 5 — Treinamento do modelo (k-Nearest Neighbors)

O **KNN** classifica um ponto novo pela **maioria** entre seus `k` vizinhos mais próximos. Aqui `k = 3`. O método `.fit()` "treina" o modelo — no caso do KNN, ele apenas **memoriza** os dados de treino.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

## Etapa 6 — Predição

Com o modelo treinado, geramos as previsões para o conjunto de teste com `.predict()` e comparamos com os rótulos reais.

In [ ]:
y_pred = knn.predict(X_test)

# Compara, lado a lado, o valor real e o previsto
comparacao = pd.DataFrame({'real': y_test.values, 'previsto': y_pred})
comparacao.head(10)

## Etapa 7 — Avaliação

Comparamos as previsões com os rótulos reais. A métrica mais simples é a **acurácia**: a fração de exemplos classificados corretamente (acertos / total).

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print('Acurácia do modelo KNN:', round(accuracy, 3))

> **A acurácia não conta a história toda.** Ela resume tudo num único número, mas não mostra *em quais espécies* o modelo erra nem se confunde uma classe com outra. Métricas mais detalhadas (**precisão**, **recall**, **F1**) e a **matriz de confusão** ficam para a **próxima aula**.

## 🚀 Próximos passos — Implantação (usar o modelo)

Um modelo aprovado pode ser **usado em dados novos**. Basta aplicar a **mesma padronização** do treino e chamar `.predict()` para uma flor nova.

In [ ]:
# Medidas de uma flor nova (cm):
# [comp. sépala, larg. sépala, comp. pétala, larg. pétala]
nova_flor = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=data.feature_names)

nova_flor_padronizada = scaler.transform(nova_flor)   # MESMA escala do treino
predicao = knn.predict(nova_flor_padronizada)

print('Espécie prevista:', data.target_names[predicao][0])